# 01 — Exploratory Data Analysis (EDA)

**Project:** Document Forgery Detector  
**Dataset:** CASIA v2  
**Date:** Day 1  

---

## Objectives

1. Verify dataset directory structure and file integrity  
2. Compute **class distribution** (Authentic vs. Tampered)  
3. Collect **image resolution statistics** (width, height, aspect ratio)  
4. Visualise **sample images** from each class  

> ⚠️ This notebook is for **exploration only**. No production logic belongs here.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import yaml
import random
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 12

In [2]:
# ── Load configuration ───────────────────────────────────────────────────────
CONFIG_PATH = Path("../configs/config.yaml")

with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

RAW_DIR = Path(cfg["dataset"]["root_dir"])
IMAGE_EXTENSIONS = set(cfg["dataset"]["image_extensions"])
CLASSES = cfg["dataset"]["classes"]

# CASIA v2 folder convention:
#   Au/ → Authentic
#   Tp/ → Tampered
CLASS_MAP = {
    "Au": "authentic",
    "Tp": "tampered",
}

print(f"Raw data directory : {RAW_DIR.resolve()}")
print(f"Expected classes   : {CLASSES}")
print(f"Image extensions   : {IMAGE_EXTENSIONS}")

Raw data directory : D:\Sem 6\Project\New folder\doc-forgery-detector\notebooks\data\raw
Expected classes   : ['authentic', 'tampered']
Image extensions   : {'.jpg', '.tif', '.png', '.bmp'}


## 1 · Dataset Inventory

In [3]:
def collect_image_paths(base_dir: Path, class_map: dict, extensions: set) -> pd.DataFrame:
    """Walk the dataset directory and return a DataFrame of (path, class, extension)."""
    records = []
    for folder, label in class_map.items():
        folder_path = base_dir / folder
        if not folder_path.exists():
            print(f"⚠️  Missing folder: {folder_path}")
            continue
        for fp in folder_path.iterdir():
            if fp.suffix.lower() in extensions:
                records.append({
                    "path": str(fp),
                    "class": label,
                    "extension": fp.suffix.lower(),
                })
    return pd.DataFrame(records)

df = collect_image_paths(RAW_DIR, CLASS_MAP, IMAGE_EXTENSIONS)
print(f"Total images found: {len(df)}")
df.head()

⚠️  Missing folder: data\raw\Au
⚠️  Missing folder: data\raw\Tp
Total images found: 0


""


## 2 · Class Distribution

In [4]:
class_counts = df["class"].value_counts()
print(class_counts)
print(f"\nImbalance ratio (tampered / authentic): {class_counts.get('tampered', 0) / max(class_counts.get('authentic', 1), 1):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
colors = ["#2ecc71", "#e74c3c"]
axes[0].bar(class_counts.index, class_counts.values, color=colors, edgecolor="black")
axes[0].set_title("Class Distribution (Count)")
axes[0].set_ylabel("Number of Images")
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 20, str(v), ha="center", fontweight="bold")

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index, autopct="%1.1f%%",
            colors=colors, startangle=90, wedgeprops={"edgecolor": "black"})
axes[1].set_title("Class Distribution (%)")

plt.tight_layout()
plt.savefig("../results/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

KeyError: 'class'

## 3 · Image Resolution Statistics

In [ ]:
def get_image_dimensions(paths: list) -> pd.DataFrame:
    """Read image dimensions without fully decoding pixel data."""
    dims = []
    corrupted = []
    for p in paths:
        try:
            with Image.open(p) as img:
                w, h = img.size
                dims.append({"width": w, "height": h, "aspect_ratio": round(w / h, 2)})
        except Exception as e:
            corrupted.append(p)
    if corrupted:
        print(f"⚠️  {len(corrupted)} corrupted/unreadable files skipped.")
    return pd.DataFrame(dims)

dims_df = get_image_dimensions(df["path"].tolist())
df = pd.concat([df.reset_index(drop=True), dims_df], axis=1)

print("\n── Resolution Summary ──")
print(df[["width", "height", "aspect_ratio"]].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df["width"], bins=50, color="#3498db", edgecolor="black", alpha=0.8)
axes[0].set_title("Width Distribution")
axes[0].set_xlabel("Pixels")

axes[1].hist(df["height"], bins=50, color="#9b59b6", edgecolor="black", alpha=0.8)
axes[1].set_title("Height Distribution")
axes[1].set_xlabel("Pixels")

axes[2].scatter(df["width"], df["height"], alpha=0.3, s=8, c="#e67e22")
axes[2].set_title("Width vs Height")
axes[2].set_xlabel("Width")
axes[2].set_ylabel("Height")

plt.tight_layout()
plt.savefig("../results/resolution_stats.png", dpi=150, bbox_inches="tight")
plt.show()

## 4 · Sample Image Visualization

In [ ]:
def show_samples(dataframe: pd.DataFrame, label: str, n: int = 5, seed: int = 42):
    """Display n random sample images for a given class label."""
    subset = dataframe[dataframe["class"] == label]
    if len(subset) == 0:
        print(f"No images found for class '{label}'.")
        return
    samples = subset.sample(n=min(n, len(subset)), random_state=seed)
    fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 4))
    if len(samples) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, samples.iterrows()):
        img = cv2.imread(row["path"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"{row['class']}\n{row['width']}×{row['height']}", fontsize=10)
        ax.axis("off")
    plt.suptitle(f"Sample Images — {label.upper()}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

show_samples(df, "authentic", n=5)
show_samples(df, "tampered", n=5)

## 5 · File Format Breakdown

In [ ]:
ext_counts = df["extension"].value_counts()
print(ext_counts)

ext_counts.plot.bar(color="#1abc9c", edgecolor="black")
plt.title("Image File Format Distribution")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## Summary

| Metric | Value |
|--------|-------|
| Total images | *run cells above* |
| Authentic count | *run cells above* |
| Tampered count | *run cells above* |
| Imbalance ratio | *run cells above* |
| Corrupted files | *run cells above* |
| Most common resolution | *run cells above* |

### Key Observations

*(Fill in after running the notebook with the actual CASIA v2 data.)*

1. **Class balance:** …  
2. **Resolution variance:** …  
3. **File formats:** …  
4. **Corrupted files:** …  

---

> **Next step (Day 2):** ELA preprocessing pipeline (`src/ela.py`).